In [23]:
import torch
import torch.nn.functional as F
import requests
import random
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import tqdm
from tqdm.auto import tqdm

In [24]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()

random.shuffle(words)
print(words[:10])
len(words)

['giuliano', 'yohannes', 'adaleah', 'terrie', 'novali', 'raeven', 'izna', 'aydrian', 'graylen', 'arhaan']


32033

In [25]:
if (device := torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")):
    print(f"Using device: {device}")

block_size = 10
epochs = 600
lr = 0.001

Using device: cuda


In [26]:
chars = ['.'] + sorted(set(''.join(words)))
def stoi(s):
    return {c: i for i, c in enumerate(chars)}[s]
def itos(i):
    return {i: c for i, c in enumerate(chars)}[i]
print(itos(1), itos(2), itos(3), itos(0))
print(stoi('.'), stoi('a'), stoi('b'), stoi('c'))

a b c .
0 1 2 3


In [27]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [28]:
X, Y = build_dataset(words)
n = X.shape[0]
Xtr, Ytr = X[:int(n*0.9)], Y[:int(n*0.9)]
Xdev, Ydev = X[int(n*0.9):], Y[int(n*0.9):]
Xtr, Ytr = [x.to(device) for x in (Xtr, Ytr)]
Xdev, Ydev = [x.to(device) for x in (Xdev, Ydev)]
print(X.shape, Y.shape)
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)

torch.Size([228146, 10]) torch.Size([228146])
torch.Size([205331, 10]) torch.Size([205331])
torch.Size([22815, 10]) torch.Size([22815])


In [29]:
trainds = torch.utils.data.TensorDataset(Xtr.cpu(), Ytr.cpu())
traindl = DataLoader(
    dataset=trainds,
    batch_size=8192,
    pin_memory=False,
    shuffle=True,
    num_workers=16,
    prefetch_factor=16
)

In [30]:
class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = torch.nn.Embedding(27,10)
        self.net = torch.nn.Sequential(
            nn.Flatten(),
            nn.Linear(100,200,False),
            nn.BatchNorm1d(200),
            nn.Tanh(),
            nn.Linear(200,27)
        )
    def forward(self, Xb):
        return self.net(self.embed(Xb))

        
        

In [ ]:
model = MLP().to(device)
# model = torch.compile(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma = 0.9999)

In [32]:
model.train()
pbar = tqdm(range(epochs), desc="train")
for e in pbar:
    for Xb, Yb in traindl:
        Xb, Yb = Xb.to(device), Yb.to(device)
        logits = model(Xb)
        loss = F.cross_entropy(logits, Yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
    pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
    # if e % 10 == 0:
    #     print(e, loss.item(), scheduler.get_last_lr()[0])

train: 100%|██████████| 600/600 [03:12<00:00,  3.11it/s, loss=1.9513, lr=4.58e-04]


In [33]:
model.eval()
with torch.no_grad():
    ix = torch.randint(0, Xdev.shape[0], (Xdev.shape[0],), device=device)
    Xb, Yb = Xdev[ix], Ydev[ix]
    logits = model(Xb)
    loss = F.cross_entropy(logits, Yb)
    print(loss.item())

2.0733563899993896


In [35]:
model.eval()
for i in range(10):
    out = []
    context = [0] * block_size
    while True:
        logits = model(torch.tensor([context], device=device))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(itos(ix))
        if ix == 0:
            break
    print(''.join(out))

maybstod.
azam.
maxe.
maccay.
letlynn.
miliana.
haidyn.
chadir.
candrice.
sonyr.
